# Dublin Bus Project

## Objective:
### Reduce an initial set of 18 problem clusters into 3 priority problem candidates
### using a simple, transparent, criteria-based selection method.

In [16]:
from pathlib import Path
import pandas as pd
import re

In [17]:
##Source note:
#The file 'DublinBusProblemMap.txt' contains an initial problem map built from sustained user-side observation.
#It systematises recurring issues noticed through frequent bus use in Dublin over more than one year,
#later expanded and refined once the project opportunity emerged.

In [18]:
with open("DublinBusProblemMap.txt", "r", encoding="utf-8") as f:
    text = f.read()
print("File loaded.")
print("Characters:", len(text))

    # DublinBusProblemMap.txt = structured problem map built 
    #from repeated first-hand observation during frequent bus use in Dublin.


File loaded.
Characters: 5835


In [28]:
# Step 1 — parse categories and atomic problems
# Method: thematic decomposition
# Why: break long text into comparable problem units

rows = []
grp = None

for line in txt.splitlines():
    line = line.strip()
    if not line:
        continue

         # detect numbered sections 
    if re.match(r"^\d+\.\s", line):
        grp = re.sub(r"^\d+\.\s*", "", line).strip()
        continue

    # skip title and central/meta labels
    if line.lower() in ["dublin bus full system problem map", "central problem", "meta problem"]:
        continue

    # atomic problem lines
    if line.startswith("-"):
        prob = line[1:].strip()
        rows.append([grp, prob])
df = pd.DataFrame(rows, columns=["grp", "prob"])
print("Atomic problems:", len(df))
print(df.head(10))

Atomic problems: 0
Empty DataFrame
Columns: [grp, prob]
Index: []


In [30]:
# Define candidate problems
# Method: reduction into project-level candidates
# Why: you do not evaluate 80+ lines directly, you evaluate candidate problems

cand = {"Service reliability and headway irregularity": ["Time & Reliability Issues","Internal Operations","Speed & Efficiency",
        "Events & Urban Context","Meta Problem"],
        
    "Real-time information failure and ghost buses": ["Ghost Buses / Information Failure","Digital Experience & Apps",
        "Communication & Transparency","Data & Information Management"],
        
    "Weak multimodal integration and transfers": ["Connections & Integration","Connectivity & Spatial Imbalance",
        "Routes & System Complexity"],
        
    "Stop infrastructure and accessibility gaps": ["Stops & Infrastructure"],
        
    "Payment friction and boarding inefficiency": ["Payment System Issues"],
        
    "Safety and onboard experience issues": ["Safety & Behaviour","Onboard Experience"]
}

In [31]:
# count evidence per candidate
# Method: evidence density
# Why: more observed issues in a cluster = stronger exploratory signal

ev = []

for name, grps in cand.items():
    n = df[df["grp"].isin(grps)].shape[0]
    ev.append([name, n])

ev = pd.DataFrame(ev, columns=["cand", "ev"])
print("\nEvidence count by candidate:")
print(ev.sort_values("ev", ascending=False))


Evidence count by candidate:
                                            cand  ev
0   Service reliability and headway irregularity   0
1  Real-time information failure and ghost buses   0
2      Weak multimodal integration and transfers   0
3     Stop infrastructure and accessibility gaps   0
4     Payment friction and boarding inefficiency   0
5           Safety and onboard experience issues   0


In [22]:
# Step 4 — build multicriteria matrix
# Method: weighted decision matrix
# Why: selection must be explicit, not intuitive only

# Criteria scale: 1 low — 5 high

mx = pd.DataFrame({
    "cand": [
        "Service reliability and headway irregularity",
        "Real-time information failure and ghost buses",
        "Weak multimodal integration and transfers",
        "Stop infrastructure and accessibility gaps",
        "Payment friction and boarding inefficiency",
        "Safety and onboard experience issues"
    ],

    # user impact
    "imp": [5, 5, 4, 3, 3, 3],

    # strategic alignment with DB priorities
    # reliable, timely, accessible, data-driven
    "ali": [5, 5, 4, 4, 3, 3],

    # measurable with available/likely data
    "mea": [5, 4, 3, 2, 3, 2],

    # feasible within your project time
    "fea": [5, 4, 3, 3, 4, 3],

    # actionable for insights/reporting
    "act": [5, 5, 4, 3, 3, 3]
})

# merge evidence
mx = mx.merge(ev, on="cand", how="left")

# convert evidence count into 1–5 scale
# simple quintile-like scaling
mx["evi"] = pd.qcut(mx["ev"].rank(method="first"), 5, labels=[1,2,3,4,5]).astype(int)

print("\nMatrix before weights:")
print(mx)


Matrix before weights:
                                            cand  imp  ali  mea  fea  act  ev  \
0   Service reliability and headway irregularity    5    5    5    5    5   0   
1  Real-time information failure and ghost buses    5    5    4    4    5   0   
2      Weak multimodal integration and transfers    4    4    3    3    4   0   
3     Stop infrastructure and accessibility gaps    3    4    2    3    3   0   
4     Payment friction and boarding inefficiency    3    3    3    4    3   0   
5           Safety and onboard experience issues    3    3    2    3    3   0   

   evi  
0    1  
1    1  
2    2  
3    3  
4    4  
5    5  


In [23]:
# Apply weights
# Method: multicriteria scoring
# Why: force a reasoned choice and make it reproducible

w = {
    "imp": 0.30,
    "mea": 0.30,
    "ali": 0.20,
    "fea": 0.10,
    "act": 0.05,
    "evi": 0.05
}

mx["score"] = (
    mx["imp"] * w["imp"] +
    mx["mea"] * w["mea"] +
    mx["ali"] * w["ali"] +
    mx["fea"] * w["fea"] +
    mx["act"] * w["act"] +
    mx["evi"] * w["evi"]
)

mx = mx.sort_values("score", ascending=False).reset_index(drop=True)

print("\nFinal ranking:")
print(mx[["cand", "score", "imp", "mea", "ali", "fea", "act", "evi", "ev"]])


Final ranking:
                                            cand  score  imp  mea  ali  fea  \
0   Service reliability and headway irregularity   4.80    5    5    5    5   
1  Real-time information failure and ghost buses   4.40    5    4    5    4   
2      Weak multimodal integration and transfers   3.50    4    3    4    3   
3     Payment friction and boarding inefficiency   3.15    3    3    3    4   
4     Stop infrastructure and accessibility gaps   2.90    3    2    4    3   
5           Safety and onboard experience issues   2.80    3    2    3    3   

   act  evi  ev  
0    5    1   0  
1    5    1   0  
2    4    2   0  
3    3    4   0  
4    3    3   0  
5    3    5   0  


In [32]:
# choose winner
# Method: decision layer
# Why: select one main problem and optional second problem
top1 = mx.iloc[0]["cand"]
top2 = mx.iloc[1]["cand"]
print("\nSelected main problem:", top1)
print("Selected secondary problem:", top2)


Selected main problem: Service reliability and headway irregularity
Selected secondary problem: Real-time information failure and ghost buses


In [25]:
# Map selected problem to metrics
# Method: traceability problem -> metric
# Why: the selected problem must become analyzable later

met = {
    "Service reliability and headway irregularity": [
        "headway variance",
        "excess waiting time",
        "early/late deviation",
        "long gap frequency",
        "bus bunching frequency"
    ],
    "Real-time information failure and ghost buses": [
        "prediction error",
        "real-time arrival mismatch",
        "missing trip frequency",
        "countdown disappearance events"
    ],
    "Weak multimodal integration and transfers": [
        "transfer waiting time",
        "missed connection risk",
        "walking-transfer burden"
    ],
    "Stop infrastructure and accessibility gaps": [
        "share of key stops without shelter",
        "share of key stops without seating",
        "screen/readability coverage"
    ],
    "Payment friction and boarding inefficiency": [
        "boarding time per passenger",
        "cash transaction delay share"
    ],
    "Safety and onboard experience issues": [
        "incident frequency",
        "cleanliness complaint rate"
    ]
}

print("\nSuggested metrics for main problem:")
for m in met[top1]:
    print("-", m)


Suggested metrics for main problem:
- headway variance
- excess waiting time
- early/late deviation
- long gap frequency
- bus bunching frequency
